In [ ]:
!pip install builtwith

  Preparing metadata (setup.py) ... done
  Created wheel for builtwith: filename=builtwith-1.3.4-py3-none-any.whl size=36077 sha256=3ba4650ca99bfece957d42fddd61df98b98fbbd7e9e05813538fecbae7b799e9
  Stored in directory: /root/.cache/pip/wheels/7f/2d/b2/606e3df914d4aeeab99c4a4e3e9a61673d2293c2e346db00c8
Successfully built builtwith


In [ ]:
import builtwith

# Analisis teknologi yang digunakan
res = builtwith.parse('https://www.detik.com/')
print(res)

{'databases': ['Firebase'], 'advertising-networks': ['Google AdSense'], 'tag-managers': ['Google Tag Manager'], 'javascript-frameworks': ['jQuery']}


## **Crawling Data**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

HEADERS = {"User-Agent": "Mozilla/5.0"}

KATEGORI_URLS = {
    "news": "https://news.detik.com/indeks",
    "finance": "https://finance.detik.com/indeks",
    "health": "https://health.detik.com/indeks",
    "sport": "https://sport.detik.com/indeks",
    "hot": "https://hot.detik.com/indeks"
}

def crawl_detik_indeks(max_pages=30, max_berita=200):
    data = {"id": [], "judul": [], "link": [], "kategori": [], "isi": []}
    idx = 0

    for kategori_loop, base_url in KATEGORI_URLS.items():
        print(f"\n=== Ambil kategori: {kategori_loop} ===")

        for page in range(1, max_pages + 1):
            if idx >= max_berita:
                break

            url = f"{base_url}?page={page}"
            try:
                r = requests.get(url, headers=HEADERS, timeout=10)
                r.raise_for_status()
            except Exception as e:
                print(f"Gagal akses {url}: {e}")
                continue

            soup = BeautifulSoup(r.text, "html.parser")
            berita_list = soup.select("h3.media__title a")

            if not berita_list:
                continue

            for berita in berita_list:
                if idx >= max_berita:
                    break

                judul = berita.get_text(strip=True)
                link = berita.get("href")

                if not link or not link.startswith("http"):
                    continue

                # ambil isi berita
                isi = ""
                try:
                    res = requests.get(link, headers=HEADERS, timeout=10)
                    res.raise_for_status()
                    soup_detail = BeautifulSoup(res.text, "html.parser")
                    paragraf = soup_detail.select("div.detail__body-text.itp_bodycontent p")
                    if not paragraf:
                        paragraf = soup_detail.select("div.detail__body-text p")

                    teks_list = []
                    for p in paragraf:
                        teks = p.get_text(" ", strip=True)
                        teks = re.sub(r"\s+", " ", teks.lower())
                        if teks:
                            teks_list.append(teks)

                    isi = " ".join(teks_list)
                except Exception:
                    isi = "(gagal ambil isi berita)"

                idx += 1
                data["id"].append(idx)
                data["judul"].append(judul)
                data["link"].append(link)
                data["kategori"].append(kategori_loop)  # gunakan kategori indeks
                data["isi"].append(isi)

                print(f"[{idx}] {judul} [{kategori_loop}]")

            if idx >= max_berita:
                break

        if idx >= max_berita:
            break

    # simpan ke CSV
    df = pd.DataFrame(data)
    df.to_csv("berita_detik_categorized.csv", index=False, encoding="utf-8-sig")
    print(f"\n[DONE] {len(df)} berita tersimpan di berita_detik_categorized.csv")
    return df


# jalankan: ambil 200 berita campur kategori sesuai indeks
crawl_detik_indeks(max_pages=30, max_berita=200)



=== Ambil kategori: news ===
[1] Alfamart Gelar Donor Darah Serentak di 34 Kota, Target 26 Ribu Kantong [news]
[2] Yusril: Presiden Tak Akan Bentuk Tim Investigasi Independen Usut Demo Ricuh [news]
[3] Bobby Nasution Wajibkan OPD di Sumut Beri Keterangan Pers Setiap Hari [news]
[4] Jelang Muktamar X, DPC PPP Se-Jateng Deklarasi Dukung Mardiono [news]
[5] Ada Diskon Tiket Whoosh untuk Keberangkatan 22-24 September, Cek Infonya! [news]
[6] Video: Profil Djamari Chaniago, Jenderal Tempur yang Kini Jadi Menko Polkam [news]
[7] Jadi Kepala Badan Komunikasi Pemerintah, Angga Raka Tetap Wamen Komdigi [news]
[8] Eropa Percepat Sanksi Energi Rusia di Tengah Tekanan Politik [news]
[9] Warga Segel Rumah Dapur MBG di Bandung karena Bau dan Beroperasi 24 Jam [news]
[10] Dunia Hari Ini: Perjanjian Militer Antara Papua Nugini dan Australia Gagal Tercapai [news]
[11] Video: Massa Demo Ojol di Depan DPR Bubar [news]
[12] SNBP 2026: Jadwal hingga Ketentuan Umum-Khusus [news]
[13] Protes Aktivis Ditahan

,id,judul,link,kategori,isi
0,1,Alfamart Gelar Donor Darah Serentak di 34 Kota...,https://news.detik.com/berita/d-8117096/alfama...,news,alfamart kembali menggelar aksi donor darah se...
1,2,Yusril: Presiden Tak Akan Bentuk Tim Investiga...,https://news.detik.com/berita/d-8117082/yusril...,news,"menteri koordinator bidang hukum, ham, imigras..."
2,3,Bobby Nasution Wajibkan OPD di Sumut Beri Kete...,https://news.detik.com/berita/d-8117080/bobby-...,news,gubernur sumatera utara (sumut) bobby nasution...
3,4,"Jelang Muktamar X, DPC PPP Se-Jateng Deklarasi...",https://news.detik.com/berita/d-8117078/jelang...,news,dukungan untuk plt ketua umum partai persatuan...
4,5,Ada Diskon Tiket Whoosh untuk Keberangkatan 22...,https://news.detik.com/berita/d-8117031/ada-di...,news,bagi pengguna whoosh rute jakarta-bandung atau...
...,...,...,...,...,...
195,196,"Patuhi Regulasi WLLP, Perusahaan Bakal Terima ...",https://news.detik.com/berita/d-8115918/patuhi...,news,kementerian ketenagakerjaan (kemnaker) menging...
196,197,Pendaftaran KJMU 2025 Tahap 2: Jadwal hingga P...,https://news.detik.com/berita/d-8115915/pendaf...,news,pemprov dki jakarta melalui dinas pendidikan j...
197,198,Netanyahu Bilang Serangan Israel 'Dibenarkan' ...,https://news.detik.com/internasional/d-8115899...,news,perdana menteri (pm) israel benjamin netanyahu...
198,199,Polda Banten Ungkap 577 Kasus Narkoba Sepanjan...,https://news.detik.com/berita/d-8115898/polda-...,news,wakapolda banten brigjen hendra wirawan menyam...
